In [47]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/jobs_cleaned.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (1167, 6)
Columns: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set', 'combined_text']


,job_id,category,job_title,job_description,job_skill_set,combined_text
0,3902668440,HR,Sr Human Resource Generalist,"SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXPERTISE ACROSS VARIOUS FUNCTIONS INCLUDING EMPLOYEE RELATIONS, TALENT ACQUISITION, PERFORMANCE MANAGEMENT, PAYROLL, COMPENSATION AND BENEFITS. THE SR. HR GENERALIST HAS RESPONSIBILITIES THAT ARE STRATE...","['employee relations', 'talent acquisition', 'performance management', 'compensation and benefits', 'employment laws and regulations', 'analytical skills', 'employee benefits administration', 'HR policies and procedures', 'recruitment strategies'...","Sr Human Resource Generalist HR SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXPERTISE ACROSS VARIOUS FUNCTIONS INCLUDING EMPLOYEE RELATIONS, TALENT ACQUISITION, PERFORMANCE MANAGEMENT, PAYROLL, COMPENSATION AND BENEFITS. THE SR. HR GENERALIST HAS ..."
1,3905823748,HR,Human Resources Manager,"BE PART OF A STELLAR TEAM AT YSB AS THE MANAGER OF HUMAN RESOURCES. JOIN US AS YSB ENTERS A NEW AND VIBRANT ERA WITH THE COMPLETION OF THE YSB CENTER FOR YOUTH SUCCESS THIS SUMMER. AS PART OF THE SENIOR LEADERSHIP TEAM AT YSB, THE HR MANAGER WILL...","['Talent Acquisition', 'Employee Performance Management', 'Legal Compliance', 'Payroll Processing', 'Microsoft Office', 'Policy Development', 'Training and Development', 'Background Checks', 'Collaboration', 'Communication', 'Problem Solving', 'L...",Human Resources Manager HR BE PART OF A STELLAR TEAM AT YSB AS THE MANAGER OF HUMAN RESOURCES. JOIN US AS YSB ENTERS A NEW AND VIBRANT ERA WITH THE COMPLETION OF THE YSB CENTER FOR YOUTH SUCCESS THIS SUMMER. AS PART OF THE SENIOR LEADERSHIP TEAM ...
2,3905854799,HR,Director of Human Resources,"OUR CLIENT IS A THRIVING ORGANIZATION OFFERING A UNIQUE OPPORTUNITY FOR A DIRECTOR OF HUMAN RESOURCES TO JOIN ITS DYNAMIC TEAM. WITH A STRONG FOCUS ON EMPLOYEE OWNERSHIP, EXCEPTIONAL BENEFITS, AND A FAMILY-ORIENTED ENVIRONMENT. OUR CLIENT IS COMM...","['Human Resources Management', 'Recruitment', 'Employee Relations', 'Performance Management', 'Compliance with Labor Laws', 'Data Management', 'Training & Development', 'Payroll Management', 'Project Management', 'Computer Skills', 'Communication...","Director of Human Resources HR OUR CLIENT IS A THRIVING ORGANIZATION OFFERING A UNIQUE OPPORTUNITY FOR A DIRECTOR OF HUMAN RESOURCES TO JOIN ITS DYNAMIC TEAM. WITH A STRONG FOCUS ON EMPLOYEE OWNERSHIP, EXCEPTIONAL BENEFITS, AND A FAMILY-ORIENTED ..."
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO)INDUSTRY: ENTERTAINMENT & MEDIA \nJOB DESCRIPTION:EAST 57TH STREET PARTNERS IS SEEKING AN EXPERIENCED AND INNOVATIVE CHIEF HUMAN RESOURCES OFFICER (CHRO) FOR A LEADING ENTERTAINMENT AND MEDIA COMPANY...,"['talent management', 'organizational development', 'HR compliance', 'HR technology systems', 'data-driven HR analytics', 'compensation and benefits strategies', 'strategic HR planning', 'workforce planning', 'employee development', 'performance ...",Chief Human Resources Officer HR JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO)INDUSTRY: ENTERTAINMENT & MEDIA \nJOB DESCRIPTION:EAST 57TH STREET PARTNERS IS SEEKING AN EXPERIENCED AND INNOVATIVE CHIEF HUMAN RESOURCES OFFICER (CHRO) FOR A LEADIN...
4,3906250451,HR,Human Resources Generalist (Hybrid Role),"DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DIGITAL ENABLEMENT SOLUTIONS PROVIDER WHO TRANSFORMS HOW PEOPLE AND TECHNOLOGY CONNECT TO ELEVATE EXPERIENCES, CREATE NEW VALUE, AND ENABLE ORGANIZATIONS TO THRIVE AND GROW. WE ARE THE LARGEST PROVIDER ...","['Microsoft Office', 'Data analysis', 'Employee onboarding', 'Strategic thinking', 'Problem solving', 'Time management', 'Attention to detail', 'Communication', 'Customer service', 'Interpersonal skills', 'Independence', 'HRIS systems', 'Organiza...","Human Resources Generalist (Hybrid Role) HR DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DIGITAL ENABLEMENT SOLUTIONS PROVIDER WHO TRANSFORMS HOW PEOPLE AND TECHNOLOGY CONNECT

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
print("combined_text" in df.columns)

True


In [50]:
df["combined_text"] = df["combined_text"].fillna("").astype(str)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = tfidf.fit_transform(df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (1167, 10000)


In [51]:
def recommend_jobs(user_skills, top_n=5):
    # Convert the user's input into a TF-IDF vector
    user_vector = tfidf.transform([user_skills])

    # Compare the user vector against every job vector
    scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Get the indices of the highest-scoring jobs
    top_indices = np.argsort(scores)[::-1][:top_n]

    # Return the recommended jobs
    results = df.iloc[top_indices][
        ["job_title", "category"]
    ].copy()

    results["similarity_score"] = scores[top_indices]

    return results

In [52]:
recommend_jobs(
    "Python SQL machine learning pandas data analysis",
    top_n=5
)

,job_title,category,similarity_score
832,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,0.231172
293,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.221847
897,Lead Software Engineer - Finance (Hybrid),FINANCE,0.197841
831,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,0.197291
372,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.195909


In [53]:
def recommend_jobs(user_skills, top_n=5):

    # Validate user input
    if not isinstance(user_skills, str) or not user_skills.strip():
        raise ValueError("Please enter your skills or interests.")

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    # Convert user input into a TF-IDF vector
    user_vector = tfidf.transform([user_skills])

    # Calculate similarity with all jobs
    scores = cosine_similarity(
        user_vector, tfidf_matrix
    ).flatten()

    # Ignore jobs with zero similarity
    matching_indices = np.where(scores > 0)[0]

    if len(matching_indices) == 0:
        return pd.DataFrame(
            columns=[
                "job_title",
                "category",
                "similarity_score"
            ]
        )

    # Sort matching jobs by similarity
    ranked_indices = matching_indices[
        np.argsort(scores[matching_indices])[::-1]
    ]

    # Select the top results
    top_indices = ranked_indices[:top_n]

    # Return useful job information
    result_columns = [
        col for col in [
            "job_id",
            "job_title",
            "category",
            "job_description",
            "job_skill_set"
        ]
        if col in df.columns
    ]

    results = df.iloc[top_indices][result_columns].copy()

    results["similarity_score"] = scores[top_indices]

    return results.reset_index(drop=True)

In [54]:
try:
    recommend_jobs("")
except ValueError as e:
    print(e)

Please enter your skills or interests.


In [55]:
# import joblib
# import os

# os.makedirs("../models", exist_ok=True)

# joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
# joblib.dump(tfidf_matrix, "../models/tfidf_matrix.pkl")

# print("Model components saved successfully!")

In [56]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the cleaned dataset
df = pd.read_csv("../data/processed/jobs_cleaned.csv")

# Safely handle missing values and convert text columns to strings
text_columns = [
    "job_title",
    "category",
    "job_description",
    "job_skill_set"
]

for column in text_columns:
    df[column] = df[column].fillna("").astype(str)

# Give greater importance to titles, categories, and skills
df["recommendation_text"] = (
    (df["job_title"] + " ") * 2
    + (df["category"] + " ") * 2
    + (df["job_skill_set"] + " ") * 2
    + df["job_description"]
)

# Convert job text into numerical TF-IDF vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df["recommendation_text"])

print("Dataset shape:", df.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

# Save the updated model components
project_dir = Path.cwd().parent
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, model_dir / "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, model_dir / "tfidf_matrix.pkl")

print("Updated TF-IDF model saved successfully!")


Dataset shape: (1167, 7)
TF-IDF matrix shape: (1167, 10000)
Vocabulary size: 10000
Updated TF-IDF model saved successfully!


In [57]:

def recommend_jobs_for_evaluation(user_skills, top_n=5):
    user_vector = tfidf.transform([user_skills])

    scores = cosine_similarity(
        user_vector, tfidf_matrix
    ).flatten()

    matching_indices = np.where(scores > 0)[0]

    ranked_indices = matching_indices[
        np.argsort(scores[matching_indices])[::-1]
    ][:top_n]

    results = df.iloc[ranked_indices][
        ["job_title", "category", "job_skill_set"]
    ].copy()

    results["similarity_score"] = scores[ranked_indices]

    return results.reset_index(drop=True)


recommend_jobs_for_evaluation(
    "Python machine learning pandas scikit-learn SQL",
    top_n=5
)


,job_title,category,job_skill_set,similarity_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...",0.247719
1,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['help desk support', 'customer service', 'networking principles', 'teamwork', 'adaptability', 'organizational skills', 'oral communication', 'written communication', 'multi-tasking', 'confidentiality', 'cultural sensitivity', 'compliance knowled...",0.224963
2,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...",0.195681
3,Information Technology Engineer,INFORMATION-TECHNOLOGY,"['SQL Server (2012+) Administration', 'Strong sense of responsibility', 'Ability to work independently', 'Good interpersonal skills', 'Communication skills', 'Self-motivated', 'Detail-minded', 'Result-oriented', 'Team player', 'Creativity', 'Exce...",0.165534
4,Enterprise Business Development Representative - US,BUSINESS-DEVELOPMENT,"['Salesforce', 'data analysis', 'lead generation', 'communication', 'team collaboration', 'client-centric approach', 'analytical skills', 'organizational skills', 'self-starter', 'multitasking', 'generative AI', 'semantic search', 'machine learni...",0.162143


In [58]:

import ast
import re
import pandas as pd

user_skills = [
    "python",
    "machine learning",
    "pandas",
    "scikit-learn",
    "sql"
]

results = recommend_jobs_for_evaluation(
    "Python machine learning pandas scikit-learn SQL",
    top_n=5
)


def parse_skills(value):
    if pd.isna(value):
        return []

    # Convert string representations of Python lists into lists
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(skill).lower() for skill in parsed]
        except (ValueError, SyntaxError):
            pass

        return [value.lower()]

    if isinstance(value, list):
        return [str(skill).lower() for skill in value]

    return []


def find_skill_matches(row):
    job_skills = parse_skills(row["job_skill_set"])

    matched = [
        skill for skill in user_skills
        if any(
            re.search(
                r"(?<!\w)" + re.escape(skill) + r"(?!\w)",
                job_skill
            )
            for job_skill in job_skills
        )
    ]

    return ", ".join(matched)


results["matched_skills"] = results.apply(
    find_skill_matches, axis=1
)

display(
    results[
        [
            "job_title",
            "category",
            "matched_skills",
            "similarity_score"
        ]
    ]
)


,job_title,category,matched_skills,similarity_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"python, machine learning, sql",0.247719
1,Information Technology Specialist,INFORMATION-TECHNOLOGY,"python, machine learning",0.224963
2,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"python, machine learning, sql",0.195681
3,Information Technology Engineer,INFORMATION-TECHNOLOGY,"machine learning, sql",0.165534
4,Enterprise Business Development Representative - US,BUSINESS-DEVELOPMENT,machine learning,0.162143


In [59]:

import ast
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def parse_job_skills(value):
    """Convert a job's skill field into a list of lowercase skills."""
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [str(skill).strip().lower() for skill in value]

    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(skill).strip().lower() for skill in parsed]
        except (ValueError, SyntaxError):
            pass

        # Fallback if the field isn't a Python-style list
        return [skill.strip().lower() for skill in value.split(",")]

    return []


def normalize_user_skills(user_input):
    """Split comma-separated user skills into individual terms."""
    return [
        skill.strip().lower()
        for skill in user_input.split(",")
        if skill.strip()
    ]


def recommend_jobs_hybrid(user_skills, top_n=5):
    if not isinstance(user_skills, str) or not user_skills.strip():
        raise ValueError("Please enter your skills.")

    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    skills = normalize_user_skills(user_skills)

    # Existing TF-IDF text similarity
    user_vector = tfidf.transform([user_skills])
    text_scores = cosine_similarity(
        user_vector, tfidf_matrix
    ).flatten()

    # Explicit matching against each job's skill list
    skill_scores = []

    for job_skill_value in df["job_skill_set"]:
        job_skills = parse_job_skills(job_skill_value)

        matched_count = sum(
            any(
                re.search(
                    r"(?<!\w)" + re.escape(skill) + r"(?!\w)",
                    job_skill
                )
                for job_skill in job_skills
            )
            for skill in skills
        )

        skill_scores.append(matched_count / len(skills))

    skill_scores = np.array(skill_scores)

    # Combine text similarity and explicit skill coverage
    final_scores = (
        0.4 * text_scores
        + 0.6 * skill_scores
    )

    # Return only jobs with some matching evidence
    matching_indices = np.where(final_scores > 0)[0]

    ranked_indices = matching_indices[
        np.argsort(final_scores[matching_indices])[::-1]
    ][:top_n]

    results = df.iloc[ranked_indices][
        ["job_title", "category", "job_skill_set"]
    ].copy()

    results["text_similarity"] = text_scores[ranked_indices]
    results["skill_coverage"] = skill_scores[ranked_indices]
    results["final_score"] = final_scores[ranked_indices]

    return results.reset_index(drop=True)


recommend_jobs_hybrid(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=5
)


,job_title,category,job_skill_set,text_similarity,skill_coverage,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...",0.247719,0.6,0.459088
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...",0.195681,0.6,0.438273
2,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['help desk support', 'customer service', 'networking principles', 'teamwork', 'adaptability', 'organizational skills', 'oral communication', 'written communication', 'multi-tasking', 'confidentiality', 'cultural sensitivity', 'compliance knowled...",0.224963,0.4,0.329985
3,Information Technology Engineer,INFORMATION-TECHNOLOGY,"['SQL Server (2012+) Administration', 'Strong sense of responsibility', 'Ability to work independently', 'Good interpersonal skills', 'Communication skills', 'Self-motivated', 'Detail-minded', 'Result-oriented', 'Team player', 'Creativity', 'Exce...",0.165534,0.4,0.306214
4,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']",0.148762,0.4,0.299505


In [60]:
print("OLD MODEL")
display(
    recommend_jobs_for_evaluation(
        "Python machine learning pandas scikit-learn SQL",
        top_n=5
    )
)

print("HYBRID MODEL")
display(
    recommend_jobs_hybrid(
        "Python, machine learning, pandas, scikit-learn, SQL",
        top_n=5
    )
)

OLD MODEL


,job_title,category,job_skill_set,similarity_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...",0.247719
1,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['help desk support', 'customer service', 'networking principles', 'teamwork', 'adaptability', 'organizational skills', 'oral communication', 'written communication', 'multi-tasking', 'confidentiality', 'cultural sensitivity', 'compliance knowled...",0.224963
2,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...",0.195681
3,Information Technology Engineer,INFORMATION-TECHNOLOGY,"['SQL Server (2012+) Administration', 'Strong sense of responsibility', 'Ability to work independently', 'Good interpersonal skills', 'Communication skills', 'Self-motivated', 'Detail-minded', 'Result-oriented', 'Team player', 'Creativity', 'Exce...",0.165534
4,Enterprise Business Development Representative - US,BUSINESS-DEVELOPMENT,"['Salesforce', 'data analysis', 'lead generation', 'communication', 'team collaboration', 'client-centric approach', 'analytical skills', 'organizational skills', 'self-starter', 'multitasking', 'generative AI', 'semantic search', 'machine learni...",0.162143


HYBRID MODEL


,job_title,category,job_skill_set,text_similarity,skill_coverage,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...",0.247719,0.6,0.459088
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...",0.195681,0.6,0.438273
2,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['help desk support', 'customer service', 'networking principles', 'teamwork', 'adaptability', 'organizational skills', 'oral communication', 'written communication', 'multi-tasking', 'confidentiality', 'cultural sensitivity', 'compliance knowled...",0.224963,0.4,0.329985
3,Information Technology Engineer,INFORMATION-TECHNOLOGY,"['SQL Server (2012+) Administration', 'Strong sense of responsibility', 'Ability to work independently', 'Good interpersonal skills', 'Communication skills', 'Self-motivated', 'Detail-minded', 'Result-oriented', 'Team player', 'Creativity', 'Exce...",0.165534,0.4,0.306214
4,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']",0.148762,0.4,0.299505


In [61]:

results = recommend_jobs_hybrid(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=5
)

for _, row in results.iterrows():
    print("=" * 70)
    print("Job:", row["job_title"])
    print("Category:", row["category"])
    print("Skills:", parse_job_skills(row["job_skill_set"]))
    print("Text similarity:", round(row["text_similarity"], 3))
    print("Skill coverage:", round(row["skill_coverage"], 3))
    print("Final score:", round(row["final_score"], 3))



Job: Lead Software Engineer - Finance (Hybrid)
Category: FINANCE
Skills: ['problem solving', 'communication', 'teamwork', 'relationship management', 'hadoop', 'aws', 'python', 'sql', 'spark', 'java', 'data modeling', 'machine learning', 'cloud computing', 'itil', 'service management', 'influencing', 'operational excellence', 'change management']
Text similarity: 0.248
Skill coverage: 0.6
Final score: 0.459
Job: VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client
Category: FINANCE
Skills: ['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series analysis', 'machine learning', 'data mining', 'tableau', 'cloud platforms', 'big data', 'statistical software', 'causal-model approaches', 'data preparation', 'regressions', 'text mining', 'aws', 'hadoop', 'apache spark', 'hive', 'pig', 's

In [62]:

import re

SKILL_ALIASES = {
    "scikit-learn": {"scikit-learn", "sklearn"},
    "machine learning": {"machine learning", "ml"},
    "python": {"python"},
    "pandas": {"pandas"},
    "sql": {"sql", "structured query language"},
}

def normalize_skill(skill):
    skill = skill.lower().strip()
    for canonical, aliases in SKILL_ALIASES.items():
        if skill in aliases:
            return canonical
    return skill


def calculate_skill_coverage(user_input, job_skill_value):
    user_skills = [
        normalize_skill(s)
        for s in user_input.split(",")
        if s.strip()
    ]

    job_skills = {
        normalize_skill(s)
        for s in parse_job_skills(job_skill_value)
    }

    if not user_skills:
        return 0.0

    matched = sum(
        skill in job_skills
        for skill in user_skills
    )

    return matched / len(set(user_skills))


In [63]:

user_input = "Python, machine learning, pandas, scikit-learn, SQL"

for _, row in results.iterrows():
    coverage = calculate_skill_coverage(
        user_input,
        row["job_skill_set"]
    )
    print(row["job_title"], "->", round(coverage, 2))


Lead Software Engineer - Finance (Hybrid) -> 0.6
VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client -> 0.6
Information Technology Specialist -> 0.4
Information Technology Engineer -> 0.0
Information Technology Specialist -> 0.4


In [64]:

import re

SKILL_ALIASES = {
    "python": {"python"},
    "sql": {"sql", "structured query language"},
    "pandas": {"pandas"},
    "scikit-learn": {"scikit-learn", "sklearn"},
    "machine learning": {"machine learning", "ml"},
}

def normalize_skill(skill):
    skill = skill.lower().strip()

    for canonical, aliases in SKILL_ALIASES.items():
        if skill in aliases:
            return canonical

    return skill


def calculate_skill_coverage(user_input, job_skill_value):
    user_skills = {
        normalize_skill(skill)
        for skill in user_input.split(",")
        if skill.strip()
    }

    job_skills = {
        normalize_skill(skill)
        for skill in parse_job_skills(job_skill_value)
    }

    if not user_skills:
        return 0.0

    return len(user_skills & job_skills) / len(user_skills)


In [65]:

user_input = "Python, machine learning, pandas, scikit-learn, SQL"

for _, row in results.iterrows():
    coverage = calculate_skill_coverage(
        user_input,
        row["job_skill_set"]
    )

    print(row["job_title"], "->", round(coverage, 2))


Lead Software Engineer - Finance (Hybrid) -> 0.6
VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client -> 0.6
Information Technology Specialist -> 0.4
Information Technology Engineer -> 0.0
Information Technology Specialist -> 0.4


In [66]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def recommend_jobs_hybrid(user_input, top_n=5):
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("Please enter your skills.")

    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    # Calculate text similarity
    user_vector = tfidf.transform([user_input])
    text_scores = cosine_similarity(
        user_vector, tfidf_matrix
    ).flatten()

    # Calculate skill coverage for each job
    skill_scores = np.array([
        calculate_skill_coverage(
            user_input, job_skill_value
        )
        for job_skill_value in df["job_skill_set"]
    ])

    # Combine the scores
    final_scores = (
        0.4 * text_scores
        + 0.6 * skill_scores
    )

    # Keep jobs with at least one matching signal
    matching_indices = np.where(
        (text_scores > 0) | (skill_scores > 0)
    )[0]

    ranked_indices = matching_indices[
        np.argsort(final_scores[matching_indices])[::-1]
    ][:top_n]

    results = df.iloc[ranked_indices][
        ["job_title", "category", "job_skill_set"]
    ].copy()

    results["text_similarity"] = text_scores[ranked_indices]
    results["skill_coverage"] = skill_scores[ranked_indices]
    results["final_score"] = final_scores[ranked_indices]

    return results.reset_index(drop=True)


In [67]:

results = recommend_jobs_hybrid(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results[
        [
            "job_title",
            "category",
            "text_similarity",
            "skill_coverage",
            "final_score"
        ]
    ].round(3)
)


,job_title,category,text_similarity,skill_coverage,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,0.248,0.6,0.459
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,0.196,0.6,0.438
2,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.225,0.4,0.330
3,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.149,0.4,0.300
4,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,0.105,0.4,0.282
5,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,0.097,0.4,0.279
6,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,0.086,0.4,0.274
7,Enterprise Business Development Representative - US,BUSINESS-DEVELOPMENT,0.162,0.2,0.185
8,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,0.139,0.2,0.176
9,Business Development Representative,BUSINESS-DEVELOPMENT,0.138,0.2,0.175


In [68]:

def get_matching_skills(user_input, job_skill_value):
    user_skills = {
        normalize_skill(skill)
        for skill in user_input.split(",")
        if skill.strip()
    }

    job_skills = {
        normalize_skill(skill)
        for skill in parse_job_skills(job_skill_value)
    }

    return sorted(user_skills & job_skills)


query = "Python, machine learning, pandas, scikit-learn, SQL"

results = recommend_jobs_hybrid(query, top_n=10)

results["matched_skills"] = results["job_skill_set"].apply(
    lambda skills: get_matching_skills(query, skills)
)

display(
    results[
        [
            "job_title",
            "category",
            "matched_skills",
            "text_similarity",
            "skill_coverage",
            "final_score",
        ]
    ].round(3)
)


,job_title,category,matched_skills,text_similarity,skill_coverage,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"[machine learning, python, sql]",0.248,0.6,0.459
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"[machine learning, python, sql]",0.196,0.6,0.438
2,Information Technology Specialist,INFORMATION-TECHNOLOGY,"[machine learning, python]",0.225,0.4,0.330
3,Information Technology Specialist,INFORMATION-TECHNOLOGY,"[python, sql]",0.149,0.4,0.300
4,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,"[python, sql]",0.105,0.4,0.282
5,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,"[python, sql]",0.097,0.4,0.279
6,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,"[python, sql]",0.086,0.4,0.274
7,Enterprise Business Development Representative - US,BUSINESS-DEVELOPMENT,[machine learning],0.162,0.2,0.185
8,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,[machine learning],0.139,0.2,0.176
9,Business Development Representative,BUSINESS-DEVELOPMENT,[machine learning],0.138,0.2,0.175


In [96]:

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# A simple title-based relevance heuristic for technical skill queries.
TECHNICAL_TERMS = {
    "python", "sql", "machine learning", "pandas",
    "scikit-learn", "sklearn", "data", "software",
    "developer", "engineering", "engineer", "analytics",
    "analyst", "technology", "programming"
}

NON_TECHNICAL_TITLE_TERMS = {
    "business development representative",
    "sales representative",
    "account executive",
    "recruiter"
}

SENIOR_TITLE_TERMS = {
    "lead", "principal", "director", "vp",
    "vice president", "chief", "head"
}



def title_relevance(job_title, user_input):
    title = str(job_title).lower()
    query = user_input.lower()

    # Role families inferred from the user's entered skills.
    ml_terms = {
        "machine learning", "scikit-learn", "sklearn",
        "pandas", "numpy", "data science"
    }

    data_titles = {
        "data scientist", "machine learning engineer",
        "ai engineer", "data analyst",
        "data engineer", "ml engineer"
    }

    software_titles = {
        "software engineer", "software developer",
        "backend developer", "python developer"
    }

    # Determine which job-title family is relevant to the query.
    if any(term in query for term in ml_terms):
        if any(term in title for term in data_titles):
            return 1.0

    if "python" in query:
        if any(term in title for term in software_titles):
            return 1.0

    # Other technical titles are still eligible, but receive
    # a neutral score rather than an automatic high score.
    technical_terms = {
        "engineer", "developer", "analyst",
        "technology", "data", "software"
    }

    if any(term in title for term in technical_terms):
        return 0.5

    return 0.25



def recommend_jobs_hybrid_v2(user_input, top_n=10):
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("Please enter your skills.")

    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    user_vector = tfidf.transform([user_input])
    text_scores = cosine_similarity(
        user_vector, tfidf_matrix
    ).flatten()

    skill_scores = np.array([
        calculate_skill_coverage(user_input, value)
        for value in df["job_skill_set"]
    ])

    title_scores = np.array([
        title_relevance(title, user_input)
        for title in df["job_title"]
    ])

    # Keep the components visible so you can inspect and tune them.
    
    title_scores = np.array([
        technical_role_relevance(title, description)
        for title, description in zip(
            df["job_title"],
            df["job_description"]
        )
    ])

    final_scores = (
        0.35 * text_scores
        + 0.45 * skill_scores
        + 0.20 * title_scores
    )
    
    # Mark pipeline/repository postings as invalid
    descriptions = df["job_description"].fillna("").astype(str).str.lower()

    pipeline_phrases = [
        "does not represent an actual job opening",
        "pipeline/repository",
        "talent pool",
        "future opportunities",
    ]

    invalid_postings = descriptions.apply(
        lambda description: any(
            phrase in description for phrase in pipeline_phrases
        )
    ).to_numpy()

    # Exclude invalid postings from ranking
    final_scores[invalid_postings] = -1

    matching_indices = np.where(
        (text_scores > 0) | (skill_scores > 0)
    )[0]

    ranked_indices = matching_indices[
        np.argsort(final_scores[matching_indices])[::-1]
    ][:top_n]

    results = df.iloc[ranked_indices][
    [
        "job_id",
        "job_title",
        "category",
        "job_skill_set",
        "job_description"
    ]
    ].copy()

    # Store the original row positions used by the recommender
    results["job_row_id"] = ranked_indices

    results["text_similarity"] = text_scores[ranked_indices]
    results["skill_coverage"] = skill_scores[ranked_indices]
    results["title_relevance"] = title_scores[ranked_indices]
    results["final_score"] = final_scores[ranked_indices]

    return results.reset_index(drop=True)


In [97]:

results_v2 = recommend_jobs_hybrid_v2(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results_v2[
        [
            "job_title",
            "category",
            "text_similarity",
            "skill_coverage",
            "title_relevance",
            "final_score"
        ]
    ].round(3)
)


,job_title,category,text_similarity,skill_coverage,title_relevance,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,0.248,0.6,1.0,0.557
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,0.196,0.6,1.0,0.538
2,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,0.105,0.4,1.0,0.417
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,0.086,0.4,1.0,0.410
4,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.149,0.4,0.6,0.352
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),INFORMATION-TECHNOLOGY,0.089,0.2,0.6,0.241
6,Information Technology Specialist II,INFORMATION-TECHNOLOGY,0.064,0.2,0.6,0.232
7,Information Technology Specialist 2 (NY HELPs) - 5138,INFORMATION-TECHNOLOGY,0.034,0.2,0.6,0.222
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,0.097,0.4,0.0,0.214
9,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,0.139,0.2,0.3,0.199


In [98]:

# Inspect the top 10 recommendations in detail

inspection = df.iloc[
    results_v2.index
].copy()

# Use the original row indices from the recommendation results
# if results_v3 contains only reset indices, map by title instead.
inspection = results_v2[
    ["job_title", "category", "job_skill_set"]
].merge(
    df[["job_title", "category", "job_description"]],
    on=["job_title", "category"],
    how="left"
)

pd.set_option("display.max_colwidth", 250)

display(
    inspection[
        [
            "job_title",
            "category",
            "job_skill_set",
            "job_description"
        ]
    ]
)


,job_title,category,job_skill_set,job_description
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...","JOB DESCRIPTION\n\nAS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LA..."
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,"['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...","JOB DESCRIPTION\n\nVP / CLIENT, FINANCE & STRATEGY DATA SCIENTIST LEAD – JPM PRIVATE CLIENT\n\nDO YOU ENJOY SOLVING KEY STRATEGIC ISSUES THAT REQUIRE ANALYTICAL APPROACHES? IF YOU HAVE A KNACK FOR BRINGING THE POWER OF DATA TO KEY STAKEHOLDERS, T..."
2,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,"['Leadership', 'Collaboration', 'Problem Solving', 'Communication', 'Continuous Improvement', 'ETL', 'ELT', 'SQL', 'NoSQL', 'Postgres', 'Spark', 'Scala', 'Java', 'Python', 'C#', 'Docker', 'Kubernetes', 'Azure', 'Powershell', 'Git', 'GraphDB', 'Bi...","GEICO IS SEEKING A STAFF DATA ENGINEER WITH A PASSION FOR BUILDING HIGH-PERFORMANCE, LOW MAINTENANCE, ZERO-DOWNTIME PLATFORMS, AND APPLICATIONS. YOU WILL HELP DRIVE OUR INSURANCE BUSINESS TRANSFORMATION AS WE TRANSITION FROM A TRADITIONAL IT MODE..."
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,"['Data Security', 'Data Management', 'Data Analysis', 'Collaboration', 'Problem Solving', 'Communication', 'Adaptability', 'Customer Focus', 'ETL', 'SQL', 'Python', 'Cloud Technologies', 'Data Warehousing', 'Test Automation', 'Data Visualization'...","WHAT YOU'LL DO\n\nWE’RE LOOKING FOR A DATA ENGINEER TO JOIN OUR ENTERPRISE DATA & ANALYTICS SPACE, SUPPORTING AND MAINTAINING DATA PRODUCTS FOR PRINCIPAL’S HUMAN RESOURCES VALUE STREAM. IN THIS ROLE, YOU’LL APPLY YOUR DATA ENGINEERING AND ANALYSI..."
4,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']","COMPANY DESCRIPTION\n ENSPYRE MANAGEMENT SERVICES IS A CERTIFIED SMALL BUSINESS BASED IN CHICAGO, IL. WE SPECIALIZE IN PROVIDING IT, PROJECT MANAGEMENT, AND ENERGY SOLUTIONS. WE HAVE A COLLABORATIVE LEADERSHIP STYLE THAT HAS LED TO OUR HISTORY OF..."
5,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']","POSITION OVERVIEWOUR INNOVATION AND TECHNOLOGY TEAM IS CONTINUALLY EXPANDING TO SUPPORT OUR STUDENTS, FACULTY AND STAFF. LOOKING TO JOIN AND INNOVATIVE TEAM THAT IS CHANGING THE WORLD? APPLY TODAY!SPECIAL NOTE: THIS JOB POSTING SERVES AS A PIPELI..."
6,Information Technology Specialist,INFORMATION-TECHNOLOGY,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']","YOUR ROLEARE YOU KNOWN FOR YOUR TECHNOLOGY SKILLS? IF SO, WE HAVE AN OPPORTUNITY THAT WOULD ALLOW YOU TO BE THE PRIMARY ON-SITE IT POINT OF CONTACT WITH INTERNAL/EXTERNAL CUSTOMERS, SALES, QUALITY AND OTHER TEAMS AS APP

In [99]:
results_v3 = recommend_jobs_hybrid_v2(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results_v3[
        [
            "job_row_id",
            "job_id",
            "job_title",
            "category",
            "text_similarity",
            "skill_coverage",
            "final_score"
        ]
    ]
)

,job_row_id,job_id,job_title,category,text_similarity,skill_coverage,final_score
0,897,3906246896,Lead Software Engineer - Finance (Hybrid),FINANCE,0.247719,0.6,0.556702
1,832,3901998589,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,0.195681,0.6,0.538489
2,757,3904956273,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,0.105174,0.4,0.416811
3,219,3903867896,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,0.085705,0.4,0.409997
4,372,3905255326,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.148762,0.4,0.352067
5,294,3906074575,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),INFORMATION-TECHNOLOGY,0.088706,0.2,0.241047
6,320,3902876526,Information Technology Specialist II,INFORMATION-TECHNOLOGY,0.063591,0.2,0.232257
7,251,3902948188,Information Technology Specialist 2 (NY HELPs) - 5138,INFORMATION-TECHNOLOGY,0.033845,0.2,0.221846
8,831,3895527225,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,0.096762,0.4,0.213867
9,233,3904383649,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,0.138982,0.2,0.198644


In [100]:
display(
    results_v3[
        [
            "job_row_id",
            "job_title",
            "job_skill_set",
            "job_description"
        ]
    ]
)

,job_row_id,job_title,job_skill_set,job_description
0,897,Lead Software Engineer - Finance (Hybrid),"['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excel...","JOB DESCRIPTION\n\nAS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LA..."
1,832,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client","['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series...","JOB DESCRIPTION\n\nVP / CLIENT, FINANCE & STRATEGY DATA SCIENTIST LEAD – JPM PRIVATE CLIENT\n\nDO YOU ENJOY SOLVING KEY STRATEGIC ISSUES THAT REQUIRE ANALYTICAL APPROACHES? IF YOU HAVE A KNACK FOR BRINGING THE POWER OF DATA TO KEY STAKEHOLDERS, T..."
2,757,Staff Data Engineer - People & Finance Tech (Remote),"['Leadership', 'Collaboration', 'Problem Solving', 'Communication', 'Continuous Improvement', 'ETL', 'ELT', 'SQL', 'NoSQL', 'Postgres', 'Spark', 'Scala', 'Java', 'Python', 'C#', 'Docker', 'Kubernetes', 'Azure', 'Powershell', 'Git', 'GraphDB', 'Bi...","GEICO IS SEEKING A STAFF DATA ENGINEER WITH A PASSION FOR BUILDING HIGH-PERFORMANCE, LOW MAINTENANCE, ZERO-DOWNTIME PLATFORMS, AND APPLICATIONS. YOU WILL HELP DRIVE OUR INSURANCE BUSINESS TRANSFORMATION AS WE TRANSITION FROM A TRADITIONAL IT MODE..."
3,219,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream","['Data Security', 'Data Management', 'Data Analysis', 'Collaboration', 'Problem Solving', 'Communication', 'Adaptability', 'Customer Focus', 'ETL', 'SQL', 'Python', 'Cloud Technologies', 'Data Warehousing', 'Test Automation', 'Data Visualization'...","WHAT YOU'LL DO\n\nWE’RE LOOKING FOR A DATA ENGINEER TO JOIN OUR ENTERPRISE DATA & ANALYTICS SPACE, SUPPORTING AND MAINTAINING DATA PRODUCTS FOR PRINCIPAL’S HUMAN RESOURCES VALUE STREAM. IN THIS ROLE, YOU’LL APPLY YOUR DATA ENGINEERING AND ANALYSI..."
4,372,Information Technology Specialist,"['SQL', 'Microsoft Suite', 'Active Directory', 'WordPress', 'Access', 'Excel', 'Customer Service', 'Analytical Skills', 'Problem Solving', 'Teamwork', 'Communication', 'Adaptability', 'Python', 'ADP', 'CellTrak', 'NetSmart', 'Litmos', 'RingCentral']","TITLE: IT SPECIALISTCOMPANY: TENDERCARE HOME HEALTH SERVICESLOCATION: INDIANAPOLIS, INDIANAHOURS: FULL-TIME, IN-OFFICE WITH REMOTE ON-CALL\nABOUT TENDERCARE HOME HEALTH SERVICES:AT TENDERCARE HOME HEALTH SERVICES, WE ARE COMMITTED TO PROVIDING OU..."
5,294,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),"['Attention to Detail', 'Self-Motivated', 'Collaboration', 'Communication', 'Problem Solving', 'Time Management', 'Adaptability', 'Oracle Database Administration', 'Disaster Recovery Procedures', 'Database Backup/Recovery Utilities', 'Recovery Ma...","ABOUT THIS POSITIONWITH MORE THAN 14,000 EMPLOYEES WORKING ACROSS 41 DEPARTMENTS, THE CITY OF PHOENIX OFFERS OPPORTUNITY FOR CAREER GROWTH AND ADVANCEMENT. WHETHER YOU'RE AN EXPERIENCED PROFESSIONAL OR JUST STARTING YOUR CAREER, WORKING IN PUBLIC..."
6,320,Information Technology Specialist II,"['Leadership', 'Organizational Skills', 'Planning Skills', 'Multi-tasking', 'Adaptability', 'Communication', 'Problem Solving', 'Time Management', 'Oracle PL/SQL', 'Java', 'Angular UI', 'Spring Boot', 'SQL', 'DevOps', 'GitHub', 'Jenkins', 'Visual...","POSITION NUMBER: 270-701-1414-XXX\nJOB CONTROL NUMBER: 427796\nTHE CALIFORNIA STUDENT AID COMMISSION (CSAC) IS SEEKING AN INFORMATION TECHNOLOGY (IT) SPECIALIST II. IN THIS ROLE, FUNCTIONS AS A SENIOR FULL STACK DEVELOPER, FOR C

In [101]:
for _, row in results_v3.iterrows():
    print("=" * 100)
    print("Row position:", row["job_row_id"])
    print("Job ID:", row["job_id"])
    print("Title:", row["job_title"])
    print("Category:", row["category"])
    print("Skills:", row["job_skill_set"])
    print("Description:", str(row["job_description"])[:1000])
    print()

Row position: 897
Job ID: 3906246896
Title: Lead Software Engineer - Finance (Hybrid)
Category: FINANCE
Skills: ['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excellence', 'Change Management']
Description: JOB DESCRIPTION

AS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LARGE SCALE DATA STORES, AND CLOUD-BASED SYSTEMS.

THE IMPACT YOU WILL MAKE

RESPONSIBILITIES

THE LEAD SOFTWARE ENGINEER - FINANCE ROLE WILL OFFER YOU THE FLEXIBILITY TO MAKE EACH DAY YOUR OWN, WHILE WORKING ALONGSIDE PEOPLE WHO CARE SO THAT YOU CAN DELIVER ON THE FOLLOWING RESPONSIBILITIES:

INDEPENDENTLY DETERMINE THE NEEDS OF THE CUSTOMER WHILE IDENTIF

In [102]:

TECHNICAL_ROLE_TERMS = [
    "software engineer",
    "software developer",
    "python developer",
    "data scientist",
    "machine learning engineer",
    "ai engineer",
    "data analyst",
    "data engineer",
    "data analytics",
    "data science",
    "backend developer",
    "frontend developer",
    "full stack developer",
    "business intelligence",
    "database administrator",
    "cloud engineer",
    "devops engineer",
    "information technology",
]

NON_TECHNICAL_ROLE_TERMS = [
    "business development representative",
    "sales representative",
    "account executive",
    "recruiter",
]


def technical_role_relevance(job_title, job_description):
    title = str(job_title).lower().strip()
    description = str(job_description).lower()

    # Reject clearly non-technical roles.
    non_technical = [
        "business development representative",
        "sales representative",
        "account executive",
        "recruiter",
    ]

    if any(term in title for term in non_technical):
        return 0.0

    # Strong title matches for data, AI/ML, and software jobs.
    strong_titles = [
        "data scientist",
        "machine learning engineer",
        "ml engineer",
        "ai engineer",
        "data analyst",
        "data engineer",
        "software engineer",
        "software developer",
        "python developer",
        "backend developer",
        "full stack developer",
        "business intelligence analyst",
    ]

    if any(term in title for term in strong_titles):
        return 1.0

    # Other technical roles get a moderate score.
    moderate_titles = [
        "database administrator",
        "devops engineer",
        "cloud engineer",
        "information technology specialist",
        "it analyst",
        "systems analyst",
        "software architect",
    ]

    if any(term in title for term in moderate_titles):
        return 0.6

    # A description alone is weaker evidence than a title.
    technical_description_terms = [
        "software development",
        "data engineering",
        "machine learning",
        "programming",
        "data analytics",
    ]

    if any(term in description for term in technical_description_terms):
        return 0.3

    return 0.0





In [103]:
results_v4 = recommend_jobs_hybrid_v2(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results_v4[
        [
            "job_row_id",
            "job_id",
            "job_title",
            "category",
            "text_similarity",
            "skill_coverage",
            "title_relevance",
            "final_score"
        ]
    ]
)

,job_row_id,job_id,job_title,category,text_similarity,skill_coverage,title_relevance,final_score
0,897,3906246896,Lead Software Engineer - Finance (Hybrid),FINANCE,0.247719,0.6,1.0,0.556702
1,832,3901998589,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",FINANCE,0.195681,0.6,1.0,0.538489
2,757,3904956273,Staff Data Engineer - People & Finance Tech (Remote),FINANCE,0.105174,0.4,1.0,0.416811
3,219,3903867896,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",HR,0.085705,0.4,1.0,0.409997
4,372,3905255326,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.148762,0.4,0.6,0.352067
5,294,3906074575,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),INFORMATION-TECHNOLOGY,0.088706,0.2,0.6,0.241047
6,320,3902876526,Information Technology Specialist II,INFORMATION-TECHNOLOGY,0.063591,0.2,0.6,0.232257
7,251,3902948188,Information Technology Specialist 2 (NY HELPs) - 5138,INFORMATION-TECHNOLOGY,0.033845,0.2,0.6,0.221846
8,831,3895527225,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,FINANCE,0.096762,0.4,0.0,0.213867
9,233,3904383649,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,0.138982,0.2,0.3,0.198644


In [104]:
results_v5 = recommend_jobs_hybrid_v2(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results_v5[
        [
            "job_row_id",
            "job_title",
            "text_similarity",
            "skill_coverage",
            "title_relevance",
            "final_score",
        ]
    ]
)

,job_row_id,job_title,text_similarity,skill_coverage,title_relevance,final_score
0,897,Lead Software Engineer - Finance (Hybrid),0.247719,0.6,1.0,0.556702
1,832,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.195681,0.6,1.0,0.538489
2,757,Staff Data Engineer - People & Finance Tech (Remote),0.105174,0.4,1.0,0.416811
3,219,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.085705,0.4,1.0,0.409997
4,372,Information Technology Specialist,0.148762,0.4,0.6,0.352067
5,294,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.088706,0.2,0.6,0.241047
6,320,Information Technology Specialist II,0.063591,0.2,0.6,0.232257
7,251,Information Technology Specialist 2 (NY HELPs) - 5138,0.033845,0.2,0.6,0.221846
8,831,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.096762,0.4,0.0,0.213867
9,233,Analyst II - Information Technology,0.138982,0.2,0.3,0.198644


In [105]:

import re

SENIORITY_TERMS = [
    "senior",
    "staff",
    "lead",
    "principal",
    "director",
    "vice president",
    "vp",
    "head of",
    "manager",
    "chief",
]

def detect_seniority(job_title):
    title = str(job_title).lower()
    pattern = r"\b(?:" + "|".join(
        re.escape(term) for term in SENIORITY_TERMS
    ) + r")\b"

    if re.search(pattern, title):
        return "Potentially senior"

    return "Seniority not indicated"


def detect_posting_validity(job_description):
    description = str(job_description).lower()

    warning_phrases = [
        "does not represent an actual job opening",
        "pipeline/repository",
        "talent pool",
        "future opportunities",
    ]

    if any(phrase in description for phrase in warning_phrases):
        return "Verify: possible pipeline posting"

    return "No pipeline warning detected"


In [106]:

# Look up the full job details using the original row positions.
review = df.iloc[
    results_v5["job_row_id"].astype(int).to_numpy()
][["job_title", "job_description"]].copy()

review["job_row_id"] = results_v5["job_row_id"].to_numpy()
review["final_score"] = results_v5["final_score"].to_numpy()

review["seniority"] = review["job_title"].apply(
    detect_seniority
)

review["posting_status"] = review["job_description"].apply(
    detect_posting_validity
)

display(
    review[
        [
            "job_row_id",
            "job_title",
            "final_score",
            "seniority",
            "posting_status",
        ]
    ].reset_index(drop=True)
)


,job_row_id,job_title,final_score,seniority,posting_status
0,897,Lead Software Engineer - Finance (Hybrid),0.556702,Potentially senior,No pipeline warning detected
1,832,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.538489,Potentially senior,No pipeline warning detected
2,757,Staff Data Engineer - People & Finance Tech (Remote),0.416811,Potentially senior,No pipeline warning detected
3,219,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.409997,Seniority not indicated,No pipeline warning detected
4,372,Information Technology Specialist,0.352067,Seniority not indicated,No pipeline warning detected
5,294,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.241047,Potentially senior,No pipeline warning detected
6,320,Information Technology Specialist II,0.232257,Seniority not indicated,No pipeline warning detected
7,251,Information Technology Specialist 2 (NY HELPs) - 5138,0.221846,Seniority not indicated,No pipeline warning detected
8,831,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.213867,Seniority not indicated,No pipeline warning detected
9,233,Analyst II - Information Technology,0.198644,Seniority not indicated,No pipeline warning detected


In [107]:

# Inspect the current top 10 recommendations
columns_to_show = [
    "job_title",
    "text_similarity",
    "skill_coverage",
    "title_relevance",
    "final_score"
]

display(
    results_v5[columns_to_show]
    .head(10)
    .reset_index(drop=True)
)


,job_title,text_similarity,skill_coverage,title_relevance,final_score
0,Lead Software Engineer - Finance (Hybrid),0.247719,0.6,1.0,0.556702
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.195681,0.6,1.0,0.538489
2,Staff Data Engineer - People & Finance Tech (Remote),0.105174,0.4,1.0,0.416811
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.085705,0.4,1.0,0.409997
4,Information Technology Specialist,0.148762,0.4,0.6,0.352067
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.088706,0.2,0.6,0.241047
6,Information Technology Specialist II,0.063591,0.2,0.6,0.232257
7,Information Technology Specialist 2 (NY HELPs) - 5138,0.033845,0.2,0.6,0.221846
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.096762,0.4,0.0,0.213867
9,Analyst II - Information Technology,0.138982,0.2,0.3,0.198644


In [108]:

# Display the jobs we need to evaluate
evaluation_df = results_v5.head(10).copy()

display(
    evaluation_df[
        ["job_title", "skill_coverage", "title_relevance", "final_score"]
    ].reset_index(drop=True)
)



,job_title,skill_coverage,title_relevance,final_score
0,Lead Software Engineer - Finance (Hybrid),0.6,1.0,0.556702
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.6,1.0,0.538489
2,Staff Data Engineer - People & Finance Tech (Remote),0.4,1.0,0.416811
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.4,1.0,0.409997
4,Information Technology Specialist,0.4,0.6,0.352067
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.2,0.6,0.241047
6,Information Technology Specialist II,0.2,0.6,0.232257
7,Information Technology Specialist 2 (NY HELPs) - 5138,0.2,0.6,0.221846
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.4,0.0,0.213867
9,Analyst II - Information Technology,0.2,0.3,0.198644


In [109]:
# Replace these example values with YOUR judgments
relevance_labels = [1, 1, 1, 2, 0, 2, 0, 2, 2, 1]

evaluation_df["relevance"] = relevance_labels

display(
    evaluation_df[
        ["job_title", "final_score", "relevance"]
    ].reset_index(drop=True)
)


,job_title,final_score,relevance
0,Lead Software Engineer - Finance (Hybrid),0.556702,1
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.538489,1
2,Staff Data Engineer - People & Finance Tech (Remote),0.416811,1
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.409997,2
4,Information Technology Specialist,0.352067,0
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.241047,2
6,Information Technology Specialist II,0.232257,0
7,Information Technology Specialist 2 (NY HELPs) - 5138,0.221846,2
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.213867,2
9,Analyst II - Information Technology,0.198644,1


In [110]:

top_5 = evaluation_df.head(5)

precision_at_5 = (
    (top_5["relevance"] == 2).sum() / len(top_5)
)

print(f"Precision@5: {precision_at_5:.2f}")


Precision@5: 0.20


In [111]:

# Inspect the first five recommendations in detail
for _, row in evaluation_df.head(5).iterrows():
    job = df.iloc[int(row["job_row_id"])]

    print("\n" + "=" * 70)
    print("JOB:", job["job_title"])
    print("YOUR RELEVANCE LABEL:", row["relevance"])
    print("REQUIRED SKILLS:", job["job_skill_set"])
    print("DESCRIPTION:", str(job["job_description"])[:700])



JOB: Lead Software Engineer - Finance (Hybrid)
YOUR RELEVANCE LABEL: 1
REQUIRED SKILLS: ['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excellence', 'Change Management']
DESCRIPTION: JOB DESCRIPTION

AS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LARGE SCALE DATA STORES, AND CLOUD-BASED SYSTEMS.

THE IMPACT YOU WILL MAKE

RESPONSIBILITIES

THE LEAD SOFTWARE ENGINEER - FINANCE ROLE WILL OFFER YOU THE FLEXIBILITY TO MAKE EACH DAY YOUR OWN, WHILE WORKING ALONGSIDE PEOPLE WHO CARE SO THAT YOU CAN DELIVER ON THE FOLLOWING RESPONSIBILITIES:

INDEPENDENTLY DETERMINE THE NEEDS OF THE CUSTOMER WHILE IDENTIFYING AND RESOLVING CONF

In [112]:

import re

PIPELINE_PHRASES = [
    "does not represent an actual job opening",
    "pipeline/repository",
    "talent pool",
    "future opportunities",
]

def is_valid_job_posting(description):
    description = str(description).lower()

    return not any(
        phrase in description
        for phrase in PIPELINE_PHRASES
    )

valid_postings = df["job_description"].apply(is_valid_job_posting)

print("Total jobs:", len(df))
print("Potential pipeline postings:", int((~valid_postings).sum()))
print("Jobs remaining after filter:", int(valid_postings.sum()))


Total jobs: 1167
Potential pipeline postings: 9
Jobs remaining after filter: 1158


In [113]:

SENIORITY_TERMS = [
    "senior",
    "staff",
    "lead",
    "principal",
    "director",
    "vice president",
    "vp",
    "head of",
    "manager",
    "chief",
]

def detect_seniority(title):
    title = str(title).lower()

    pattern = r"\b(?:" + "|".join(
        re.escape(term) for term in SENIORITY_TERMS
    ) + r")\b"

    if re.search(pattern, title):
        return "Potentially senior"

    return "Seniority not indicated"

df["seniority_flag"] = df["job_title"].apply(detect_seniority)

display(
    df[["job_title", "seniority_flag"]].head(10)
)


,job_title,seniority_flag
0,Sr Human Resource Generalist,Seniority not indicated
1,Human Resources Manager,Potentially senior
2,Director of Human Resources,Potentially senior
3,Chief Human Resources Officer,Potentially senior
4,Human Resources Generalist (Hybrid Role),Seniority not indicated
5,Human Resources Manager,Potentially senior
6,Human Resources Generalist,Seniority not indicated
7,Human Resources Information System Specialist,Seniority not indicated
8,Human Resources Generalist,Seniority not indicated
9,Human Resources Project Manager,Potentially senior


In [114]:

results_v6 = recommend_jobs_hybrid_v2(
    "Python, machine learning, pandas, scikit-learn, SQL",
    top_n=10
)

display(
    results_v6[
        [
            "job_title",
            "text_similarity",
            "skill_coverage",
            "title_relevance",
            "final_score",
        ]
    ].reset_index(drop=True)
)


,job_title,text_similarity,skill_coverage,title_relevance,final_score
0,Lead Software Engineer - Finance (Hybrid),0.247719,0.6,1.0,0.556702
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.195681,0.6,1.0,0.538489
2,Staff Data Engineer - People & Finance Tech (Remote),0.105174,0.4,1.0,0.416811
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.085705,0.4,1.0,0.409997
4,Information Technology Specialist,0.148762,0.4,0.6,0.352067
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.088706,0.2,0.6,0.241047
6,Information Technology Specialist II,0.063591,0.2,0.6,0.232257
7,Information Technology Specialist 2 (NY HELPs) - 5138,0.033845,0.2,0.6,0.221846
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.096762,0.4,0.0,0.213867
9,Analyst II - Information Technology,0.138982,0.2,0.3,0.198644


In [115]:

# Replace these example labels with your judgments,
# in exactly the order of results_v6.
relevance_labels_v6 = [2, 1, 2, 1, 0, 2, 1, 0, 2, 1]

evaluation_v6 = results_v6.head(10).copy()
evaluation_v6["relevance"] = relevance_labels_v6

display(
    evaluation_v6[
        ["job_title", "final_score", "relevance"]
    ].reset_index(drop=True)
)


,job_title,final_score,relevance
0,Lead Software Engineer - Finance (Hybrid),0.556702,2
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.538489,1
2,Staff Data Engineer - People & Finance Tech (Remote),0.416811,2
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.409997,1
4,Information Technology Specialist,0.352067,0
5,Senior Oracle Database Administrator (Senior Information Technology Systems Specialist),0.241047,2
6,Information Technology Specialist II,0.232257,1
7,Information Technology Specialist 2 (NY HELPs) - 5138,0.221846,0
8,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.213867,2
9,Analyst II - Information Technology,0.198644,1


In [116]:

top_5_v6 = evaluation_v6.head(5)

precision_at_5_v6 = (
    (top_5_v6["relevance"] == 2).sum() / len(top_5_v6)
)

print(f"Old Precision@5: 0.20")
print(f"New Precision@5: {precision_at_5_v6:.2f}")


Old Precision@5: 0.20
New Precision@5: 0.40


In [117]:

test_queries = [
    "Python, machine learning, pandas, scikit-learn, SQL",
    "SQL, Excel, Power BI, data visualization, statistics",
    "Python, ETL, SQL, data warehousing, cloud technologies",
]

for query in test_queries:
    print("\n" + "=" * 70)
    print("USER SKILLS:", query)

    results = recommend_jobs_hybrid_v2(query, top_n=5)

    display(
        results[
            ["job_title", "skill_coverage", "final_score"]
        ].reset_index(drop=True)
    )



USER SKILLS: Python, machine learning, pandas, scikit-learn, SQL


,job_title,skill_coverage,final_score
0,Lead Software Engineer - Finance (Hybrid),0.6,0.556702
1,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.6,0.538489
2,Staff Data Engineer - People & Finance Tech (Remote),0.4,0.416811
3,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.4,0.409997
4,Information Technology Specialist,0.4,0.352067



USER SKILLS: SQL, Excel, Power BI, data visualization, statistics


,job_title,skill_coverage,final_score
0,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.4,0.428378
1,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.4,0.415005
2,Information Technology Programmer,0.6,0.395691
3,Information Technology Developer,0.6,0.330922
4,Information Technology Specialist,0.4,0.320014



USER SKILLS: Python, ETL, SQL, data warehousing, cloud technologies


,job_title,skill_coverage,final_score
0,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",1.0,0.741022
1,Staff Data Engineer - People & Finance Tech (Remote),0.6,0.523170
2,Data Solutions Analyst II (Managed Care - Finance Focus) - R7269,0.8,0.462211
3,Lead Software Engineer - Finance (Hybrid),0.4,0.430987
4,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.4,0.420491


In [120]:

test_queries = {
    "ML": "Python, machine learning, pandas, scikit-learn, SQL",
    "Analytics": "SQL, Excel, Power BI, data visualization, statistics",
    "Data Engineering": "Python, ETL, SQL, data warehousing, cloud technologies",
}

evaluation_rows = []

for profile, query in test_queries.items():
    results = recommend_jobs_hybrid_v2(query, top_n=5)

    for _, row in results.iterrows():
        evaluation_rows.append({
            "profile": profile,
            "job_title": row["job_title"],
            "final_score": row["final_score"],
            "relevance": None,
        })

evaluation_tests = pd.DataFrame(evaluation_rows)

display(evaluation_tests)


,profile,job_title,final_score,relevance
0,ML,Lead Software Engineer - Finance (Hybrid),0.556702,None
1,ML,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.538489,None
2,ML,Staff Data Engineer - People & Finance Tech (Remote),0.416811,None
3,ML,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.409997,None
4,ML,Information Technology Specialist,0.352067,None
5,Analytics,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.428378,None
6,Analytics,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.415005,None
7,Analytics,Information Technology Programmer,0.395691,None
8,Analytics,Information Technology Developer,0.330922,None
9,Analytics,Information Technology Specialist,0.320014,None


In [122]:

for _, row in evaluation_tests.iterrows():
    matching_jobs = df[
        df["job_title"] == row["job_title"]
    ]

    print("\n" + "=" * 60)
    print("PROFILE:", row["profile"])
    print("JOB:", row["job_title"])

    if matching_jobs.empty:
        print("No matching job found in df.")
    else:
        job = matching_jobs.iloc[0]
        print("SKILLS:", job["job_skill_set"])
        print("DESCRIPTION:", str(job["job_description"])[:500])



PROFILE: ML
JOB: Lead Software Engineer - Finance (Hybrid)
SKILLS: ['Problem Solving', 'Communication', 'Teamwork', 'Relationship Management', 'Hadoop', 'AWS', 'Python', 'SQL', 'Spark', 'Java', 'Data Modeling', 'Machine Learning', 'Cloud Computing', 'ITIL', 'Service Management', 'Influencing', 'Operational Excellence', 'Change Management']
DESCRIPTION: JOB DESCRIPTION

AS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LARGE SCALE DATA STORES, AND CLOUD-BASED SYSTEMS.

THE IMPACT YOU WILL MAKE

RESPONSIBILITIES

THE LEAD SOFTWARE ENGINEER - FINANCE ROLE WILL OFFER YOU THE FLEXIBILITY TO MAKE EACH DAY YOUR OWN, WHILE WORKING ALONGSIDE PEOPLE WHO CARE SO THAT YOU CAN DELIVER

PROFILE: ML
JOB: VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client
SKILLS: ['data integrity', 'communication', 'stakeho

In [123]:

for profile, group in evaluation_tests.groupby("profile", sort=False):
    top_5 = group.head(5)

    precision = (top_5["relevance"] == 2).mean()

    print(f"{profile} Precision@5: {precision:.2f}")


ML Precision@5: 0.00
Analytics Precision@5: 0.00
Data Engineering Precision@5: 0.00


In [124]:
display(
    evaluation_tests[
        ["profile", "job_title", "final_score", "relevance"]
    ]
)

,profile,job_title,final_score,relevance
0,ML,Lead Software Engineer - Finance (Hybrid),0.556702,None
1,ML,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.538489,None
2,ML,Staff Data Engineer - People & Finance Tech (Remote),0.416811,None
3,ML,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.409997,None
4,ML,Information Technology Specialist,0.352067,None
5,Analytics,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",0.428378,None
6,Analytics,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",0.415005,None
7,Analytics,Information Technology Programmer,0.395691,None
8,Analytics,Information Technology Developer,0.330922,None
9,Analytics,Information Technology Specialist,0.320014,None


In [125]:
# Temporary labels — replace these with your actual judgments.
relevance_labels = [
    1, 1, 1, 2, 0,  # ML: first 5 jobs
    1, 1, 1, 1, 0,  # Analytics: first 5 jobs
    2, 1, 1, 1, 1   # Data Engineering: first 5 jobs
]

evaluation_tests["relevance"] = relevance_labels

display(
    evaluation_tests[
        ["profile", "job_title", "relevance"]
    ]
)

,profile,job_title,relevance
0,ML,Lead Software Engineer - Finance (Hybrid),1
1,ML,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",1
2,ML,Staff Data Engineer - People & Finance Tech (Remote),1
3,ML,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",2
4,ML,Information Technology Specialist,0
5,Analytics,"VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client",1
6,Analytics,"Data Engineer - Enterprise Data and Analytics, Human Resources Stream",1
7,Analytics,Information Technology Programmer,1
8,Analytics,Information Technology Developer,1
9,Analytics,Information Technology Specialist,0


In [126]:
for profile, group in evaluation_tests.groupby(
    "profile", sort=False
):
    top_5 = group.head(5)

    precision_at_5 = (
        (top_5["relevance"] == 2).sum() / 5
    )

    print(
        f"{profile} Precision@5: "
        f"{precision_at_5:.2f} "
        f"({int((top_5['relevance'] == 2).sum())}/5 relevant)"
    )

ML Precision@5: 0.20 (1/5 relevant)
Analytics Precision@5: 0.00 (0/5 relevant)
Data Engineering Precision@5: 0.20 (1/5 relevant)


In [127]:

analytics_query = (
    "SQL, Excel, Power BI, data visualization, statistics"
)

analytics_results = recommend_jobs_hybrid_v2(
    analytics_query,
    top_n=5
)

for _, row in analytics_results.iterrows():
    print("\n" + "=" * 60)
    print("JOB:", row["job_title"])
    print("SKILL COVERAGE:", row["skill_coverage"])
    print("FINAL SCORE:", row["final_score"])

    matching = df[
        df["job_title"] == row["job_title"]
    ]

    if not matching.empty:
        job = matching.iloc[0]
        print("REQUIRED SKILLS:", job["job_skill_set"])
        print("DESCRIPTION:", str(job["job_description"])[:700])



JOB: VP / Client, Finance & Strategy Data Scientist Lead – JPM Private Client
SKILL COVERAGE: 0.4
FINAL SCORE: 0.42837757980959273
REQUIRED SKILLS: ['data integrity', 'communication', 'stakeholder engagement', 'strategic thinking', 'problem solving', 'project management', 'teamwork', 'business acumen', 'transparency', 'python', 'sql', 'data visualization', 'predictive modeling', 'time series analysis', 'machine learning', 'data mining', 'tableau', 'cloud platforms', 'big data', 'statistical software', 'causal-model approaches', 'data preparation', 'regressions', 'text mining', 'aws', 'hadoop', 'apache spark', 'hive', 'pig', 'storytelling', 'iterative development']
DESCRIPTION: JOB DESCRIPTION

VP / CLIENT, FINANCE & STRATEGY DATA SCIENTIST LEAD – JPM PRIVATE CLIENT

DO YOU ENJOY SOLVING KEY STRATEGIC ISSUES THAT REQUIRE ANALYTICAL APPROACHES? IF YOU HAVE A KNACK FOR BRINGING THE POWER OF DATA TO KEY STAKEHOLDERS, THAT CAN CHANGE THE GAME, THEN THIS OPPORTUNITY IS FOR YOU!

JOB SUMMARY